In [1]:
# kaufman_efficiency_ratio ER
import pandas as pd
import numpy as np

def kaufman_efficiency_ratio(close_series, period=10):
    # Variation nette sur la période (entre maintenant et il y a N périodes)
    net_change = np.abs(close_series - close_series.shift(period))
    
    # Somme des variations absolues bar à bar sur la période
    change = np.abs(close_series - close_series.shift(1))
    volatility = change.rolling(window=period).sum()
    
    # Éviter la division par zéro
    er = np.where(volatility == 0, 0, net_change / volatility)
    
    return pd.Series(er, index=close_series.index)

# Exemple d'utilisation :
# df['KER'] = kaufman_efficiency_ratio(df['Close'], period=10)

In [2]:
# swing structure
import numpy as np
import pandas as pd


def implement_market_structure_states_strict(df, n=5):
    """Calcule les états du marché avec transition obligatoire par l'état NEUTRAL.

    Aucun passage direct entre BULLISH et BEARISH n'est autorisé.
    """
    df = df.copy()
    length = len(df)

    df['is_swing_high'] = False
    df['is_swing_low'] = False
    for k in range(n, len(df) - n):
        # Le plus haut des n bougies précédentes et suivantes
        high_range = df['High'].iloc[k - n:k + n + 1]
        if df['High'].iloc[k] == high_range.max():
            df.loc[df.index[k], 'is_swing_high'] = True
            
        # Le plus bas des n bougies précédentes et suivantes
        low_range = df['Low'].iloc[k - n:k + n + 1]
        if df['Low'].iloc[k] == low_range.min():
            df.loc[df.index[k], 'is_swing_low'] = True

    states = []
    current_state = "NEUTRAL"

    # Variables de structure de marché
    last_confirmed_high = None
    last_confirmed_low = None
    higher_low = None  # Le plancher (uniquement actif en BULLISH)
    lower_high = None  # Le plafond (uniquement actif en BEARISH)

    for j in range(length):
        close_p = df["Close"].iloc[j]

        # --- ÉTAPE B : Machine à états avec transit strict par NEUTRAL ---
        if current_state == "BULLISH":
            # Sortie de Bullish : Clôture sous le Higher Low (HL) -> NEUTRAL uniquement
            if higher_low is not None and close_p < higher_low:
                current_state = "NEUTRAL"
                last_confirmed_high = None
                last_confirmed_low = None
                higher_low = None  # Reset des repères obsolètes
                lower_high = None

        elif current_state == "BEARISH":
            # Sortie de Bearish : Clôture au-dessus du Lower High (LH) -> NEUTRAL uniquement
            if lower_high is not None and close_p > lower_high:
                current_state = "NEUTRAL"
                last_confirmed_high = None
                last_confirmed_low = None
                higher_low = None
                lower_high = None  # Reset des repères obsolètes

        elif current_state == "NEUTRAL":
            # En zone neutre, on attend un NOUVEAU BOS pour définir la tendance
            # Entrée en Bullish : Clôture au-dessus du dernier sommet confirmé
            # il faut tjs un high and un low avant de former une structure bearish ou bullish
            if last_confirmed_high is not None and last_confirmed_low is not None and close_p > last_confirmed_high:
                current_state = "BULLISH"
                higher_low = (
                    last_confirmed_low  # On fige le dernier creux comme HL
                )

            # Entrée en Bearish : Clôture sous le dernier creux confirmé
            elif last_confirmed_high is not None and last_confirmed_low is not None and close_p < last_confirmed_low:
                current_state = "BEARISH"
                lower_high = (
                    last_confirmed_high  # On fige le dernier sommet comme LH
                )

        states.append(current_state)

         # --- ÉTAPE A : Enregistrement des Swings dès qu'ils se confirment ---
        if df["is_swing_high"].iloc[j] == True:
            last_confirmed_high = df["High"].iloc[j]
            if current_state == "BEARISH" and lower_high > last_confirmed_high:
                lower_high = last_confirmed_high

        if df["is_swing_low"].iloc[j] == True:
            last_confirmed_low = df["Low"].iloc[j]
            if current_state == "BULLISH" and higher_low < last_confirmed_low:
                higher_low = last_confirmed_low

    df["market_state"] = states
    return df


In [3]:
# range box
import numpy as np

def is_range_box(highs, lows, closes, N=20,
                  touch_tol_pct=0.10, min_touches=2,
                  overlap_threshold=0.60, position_threshold=0.25):
    """
    highs, lows, closes: mảng giá Daily ĐÃ ĐÓNG, phần tử cuối cùng là nến gần nhất
                          (không bao gồm nến Daily đang chạy dở, tránh lookahead).
    Trả về: (is_range: bool, details: dict)
    """
    h = np.array(highs[-N:])
    l = np.array(lows[-N:])
    c = closes.iloc[-1]

    box_high = h.max()
    box_low = l.min()
    box_height = box_high - box_low
    if box_height <= 0:
        return False, {}

    # 1. Overlap giữa các nến liên tiếp
    overlaps = 0
    for i in range(1, N):
        if l[i] <= h[i-1] and h[i] >= l[i-1]:
            overlaps += 1
    overlap_ratio = overlaps / (N - 1)

    # 2. Số lần chạm biên trên / dưới
    tol = box_height * touch_tol_pct
    upper_touches = int(np.sum(h >= box_high - tol))
    lower_touches = int(np.sum(l <= box_low + tol))

    # 3. Vị trí close hiện tại so với trung điểm box
    mid = (box_high + box_low) / 2
    position_ratio = abs(c - mid) / box_height

    is_range = (
        overlap_ratio >= overlap_threshold
        and upper_touches >= min_touches
        and lower_touches >= min_touches
        and position_ratio <= position_threshold
    )

    details = {
        "box_high": box_high, "box_low": box_low, "box_height": box_height,
        "overlap_ratio": round(overlap_ratio, 3),
        "upper_touches": upper_touches, "lower_touches": lower_touches,
        "position_ratio": round(position_ratio, 3),
    }
    return is_range, details

In [4]:
# rsi ranging stratey
from backtesting import Strategy

global_tp_multiple_rsi = 0
global_stop_trade_rsi = 0
global_start_trade_hour = 0
global_er15_threshold = 0

class OpeningRangeBreakout(Strategy):
    # variables
    tp_multiple_rsi = 5
    start_trade_hour = 5
    stop_trade_rsi = 16
    er15_threshold = 0.3

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.current_day_order_placed = False
        self.last_low = None
        self.last_high = None
        self.order_placed_hour = None
        self.order_executed = False

        if global_tp_multiple_rsi != 0:
            self.tp_multiple_rsi = global_tp_multiple_rsi

        if global_stop_trade_hour != 0:
            self.stop_trade_rsi = global_stop_trade_rsi

        if global_start_trade_hour != 0:
            self.start_trade_hour = global_start_trade_hour

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.current_day_order_placed is True and self.order_placed_hour:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()

        self.current_day = day
        self.current_day_order_placed = False
        self.order_placed_hour = None
        self.last_low = None
        self.last_high = None
        self.order_executed = False

    def get_tag(self):
        return f"{self.current_day} {self.order_placed_hour}"

    def place_long_order(self, candle_date):
        planned_entry_price = self.data.High[-1]
        sl_price = self.last_low
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple_rsi = self.tp_multiple_rsi 
        tp_price = planned_entry_price + per_share_risk * tp_multiple_rsi
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        # print(candle_date)
        # print(f"long per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go long limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.buy(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
            self.current_day_order_placed = True
        
    def place_short_order(self, candle_date):
        planned_entry_price = self.data.Low[-1]
        sl_price = self.last_high
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple_rsi = self.tp_multiple_rsi 
        tp_price = planned_entry_price - per_share_risk * tp_multiple_rsi
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        # print(candle_date)
        # print(f"short per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go short limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.sell(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
            self.current_day_order_placed = True
            
    def use_rsi_entry_strategy_market_ranging(self, candle_datetime):
        c_long = self.data['RSI14'][-1] < 35
        c_short = self.data['RSI14'][-1] > 65

        if not self.current_day_order_placed:
            # print(f"not placed RSI14 = {self.data['RSI14'][-1]}")
            if c_long:
                self.place_long_order(candle_datetime)

            if c_short:
                self.place_short_order(candle_datetime)
        else:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            # print(f"placed RSI14 = {self.data['RSI14'][-1]}")
            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()
                        self.current_day_order_placed = False
                        if order.is_long:
                            self.place_long_order(candle_datetime)
                        else:
                            self.place_short_order(candle_datetime)

    def next(self):
        candle_datetime = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date or candle_datetime.time().hour >= self.stop_trade_rsi:
            self.reset_range(current_bar_date)

        if self.last_low is None:
            self.last_low = self.data.Low[-1]
        else:
            if self.last_low > self.data.Low[-1]:
                self.last_low = self.data.Low[-1]

        if self.last_high is None:
            self.last_high = self.data.High[-1]
        else:
            if self.last_high < self.data.High[-1]:
                self.last_high = self.data.High[-1]

        c_market_structure = self.data['market_state'][-1] == 'NEUTRAL'
        c_range_box = self.data['is_range_box'][-1] == True
        c_er = self.data['ER'][-1] < self.er15_threshold
        c_time_limit = candle_datetime.time().hour < self.stop_trade_rsi and candle_datetime.time().hour > self.start_trade_hour
        
        if c_market_structure and c_er and c_range_box and c_time_limit:
            # print(candle_datetime)
            # print(f"c_market_structure = {self.data['market_state'][-1]} & c_er = {self.data['ER'][-1]}")
            self.use_rsi_entry_strategy_market_ranging(candle_datetime)

        # if self.current_day_order_placed == True:
        #     print(candle_datetime)
        #     print(self.orders)
        #     print(self.trades)
        # print("--------------------------------")
            



c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [5]:
# orb trending strategy
import pandas as pd
import time

from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

global_range_to_consider = 0
global_tp_multiple_orb = 0
global_stop_trade_orb = 0
global_close_range_hour = 0
global_er15_threshold = 0

class OpeningRangeBreakout(Strategy):
    # variables
    range_to_consider = 3
    tp_multiple_orb = 5.5
    stop_trade_orb = 18
    close_range_hour = 8
    er15_threshold = 0.3

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000
    open_range_hour = close_range_hour - range_to_consider

    # def init(self, range_to_consider, tp_multiple_orb, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.opening_range_high = None
        self.opening_range_low = None
        if global_range_to_consider != 0:
            self.range_to_consider = global_range_to_consider

        if global_tp_multiple_orb != 0:
            self.tp_multiple_orb = global_tp_multiple_orb

        if global_stop_trade_orb != 0:
            self.stop_trade_orb = global_stop_trade_orb

        if global_close_range_hour != 0:
            self.close_range_hour = global_close_range_hour

        if global_close_range_hour != 0 and global_range_to_consider != 0:
            self.open_range_hour = global_close_range_hour - global_range_to_consider

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

    def reset_range(self, day):
        self.current_day = day
        self.current_day_position_taked = False
        self.opening_range_high = None
        self.opening_range_low = None

    def get_position_size(self, entry_price: float, stop_price : float) -> int:
        per_share_risk = abs(entry_price - stop_price)
        print(per_share_risk)
        if per_share_risk == 0:
            return 0

        share_by_risk = (self.risk_percent * self.equity) / entry_price

        share_by_leverage = (self.max_leverage * self.equity) / entry_price
        print(f"share_by_risk = {share_by_risk} & share_by_leverage = {share_by_leverage}")

        return int(min(share_by_risk, share_by_leverage))

    def next(self):
        t = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()

        if (self.current_day != current_bar_date):
            self.reset_range(current_bar_date)
            # print(f"new date {current_bar_date}")

        if self.open_range_hour < t.time().hour and t.time().hour <= self.close_range_hour:
            if self.opening_range_high is not None:
                self.opening_range_high = max(self.opening_range_high, self.data.High[-1])
            else:
                self.opening_range_high = self.data.High[-1]

            if self.opening_range_low is not None:
                    self.opening_range_low = min(self.opening_range_low, self.data.Low[-1])
            else:
                self.opening_range_low = self.data.Low[-1]

            if t.time().hour < self.close_range_hour:
                return

        if t.time().hour > self.close_range_hour and self.opening_range_high is not None and self.opening_range_low is not None:
            c_long = self.data.Open[-1] < self.opening_range_high and self.data.Close[-1] > self.opening_range_high and self.data['market_state'][-1] == 'BULLISH'
            c_short = self.data.Open[-1] > self.opening_range_low and self.data.Close[-1] < self.opening_range_low and self.data['market_state'][-1] == 'BEARISH'

            c_range_box = self.data['is_range_box'][-1] == False
            
            c2 = (self.opening_range_high - self.opening_range_low) < self.data['ATR'][-1] * 2.5
            c3 = (self.opening_range_high - self.opening_range_low) > self.data['ATR'][-1] * 1.0
            c4 = t.time().hour < self.stop_trade_orb
            c5 = self.data['ER'][-1] > self.er15_threshold

            if not self.current_day_position_taked and c2 and c3 and c4 and c5 and c_range_box:
                if c_long:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_low
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_price = planned_entry_price + per_share_risk * self.tp_multiple_orb
                    position_size = int((self.risk_percent * self.equity) / per_share_risk)
                    # position_size = round((self.risk_percent * self.equity) / (per_share_risk * 100 * 6.35), 2)
                    # if position_size > 1:
                    #     position_size = 1
                    # print(f"position_size = {position_size} & self.risk_percent = {self.risk_percent * self.equity} & per_share_risk = {per_share_risk}")

                    # print(f'Go long tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.buy(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_position_taked = True
                if c_short:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_high
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_price = planned_entry_price - per_share_risk * self.tp_multiple_orb
                    position_size = int((self.risk_percent * self.equity) / per_share_risk)
                    # if position_size > 1:
                    #     position_size = 1
                    # print(f"position_size = {position_size} & self.risk_percent = {self.risk_percent * self.equity} & per_share_risk = {per_share_risk}")
                    # print(f'Go short tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.sell(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_position_taked = True
        # if t.time().hour > self.close_range_hour and self.opening_range_high is not None and self.opening_range_low is not None:
        #     c_long = self.data.Open[-1] < self.opening_range_high and self.data.Close[-1] > self.opening_range_high and self.data['market_state'] == 'BULLISH'
        #     c_short = self.data.Open[-1] > self.opening_range_low and self.data.Close[-1] < self.opening_range_low and self.data['market_state'] == 'BEARISH'
            
        #     c2 = (self.opening_range_high - self.opening_range_low) < self.data['ATR'][-1] * 2.5
        #     c3 = (self.opening_range_high - self.opening_range_low) > self.data['ATR'][-1] * 1.0
        #     c4 = t.time().hour < self.stop_trade_orb
        #     c5 = self.data['ER'][-1] > self.er15_threshold

        #     if not self.current_day_position_taked:
        #         if c_long and c2 and c3 and c4 and c5:
        #             planned_entry_price = self.data.Close[-1]
        #             sl_price = self.opening_range_low
        #             per_share_risk = abs(planned_entry_price - sl_price)
        #             tp_price = planned_entry_price + per_share_risk * self.tp_multiple_orb
        #             position_size = int((self.risk_percent * self.equity) / per_share_risk)

        #             # print(f'Go long tp = {tp_price} sl = {sl_price} position size = {position_size}')
        #             order = self.buy(size = position_size, tp=tp_price, sl=sl_price)
        #             self.current_day_position_taked = True
        #         if c_short and c2 and c3 and c4 and c5:
        #             planned_entry_price = self.data.Close[-1]
        #             sl_price = self.opening_range_high
        #             per_share_risk = abs(planned_entry_price - sl_price)
        #             tp_price = planned_entry_price - per_share_risk * self.tp_multiple_orb
        #             position_size = int((self.risk_percent * self.equity) / per_share_risk)

        #             # print(f'Go short tp = {tp_price} sl = {sl_price} position size = {position_size}')
        #             order = self.sell(size = position_size, tp=tp_price, sl=sl_price)
        #             self.current_day_position_taked = True

In [6]:
# rsi and orb
from backtesting import Strategy

# global_tp_multiple = 0
global_stop_trade_hour = 0
global_start_trade_hour = 0
global_er15_threshold = 0
global_range_to_consider = 0
global_close_range_hour = 0

class OpeningRangeBreakout(Strategy):
    # variables
    range_to_consider = 3
    tp_multiple_orb = 5
    tp_multiple_rsi = 4
    start_trade_hour = 5
    stop_trade_hour = 16
    er15_threshold = 0.3
    close_range_hour = 8

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000
    open_range_hour = close_range_hour - range_to_consider

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.current_day_order_placed = False
        self.last_low = None
        self.last_high = None
        self.order_placed_hour = None
        self.order_executed = False
        self.opening_range_high = None
        self.opening_range_low = None

        if global_range_to_consider != 0:
                    self.range_to_consider = global_range_to_consider

        # if global_tp_multiple != 0:
        #     self.tp_multiple = global_tp_multiple

        if global_stop_trade_hour != 0:
            self.stop_trade_hour = global_stop_trade_hour

        if global_start_trade_hour != 0:
            self.start_trade_hour = global_start_trade_hour

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

        if global_close_range_hour != 0:
                    self.close_range_hour = global_close_range_hour
            
        if global_close_range_hour != 0 and global_range_to_consider != 0:
            self.open_range_hour = global_close_range_hour - global_range_to_consider

    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.current_day_order_placed is True and self.order_placed_hour:
            for trade in self.trades:
                if trade.tag == self.order_placed_hour:
                    self.order_executed = True

            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.order_placed_hour:
                        order.cancel()

        self.current_day = day
        self.current_day_order_placed = False
        self.opening_range_high = None
        self.opening_range_low = None
        self.order_placed_hour = None
        self.last_low = None
        self.last_high = None
        self.order_executed = False

    def get_tag(self):
            return f"{self.current_day} {self.order_placed_hour}"
    
    def place_long_order(self, candle_date):
        planned_entry_price = self.data.High[-1]
        sl_price = self.last_low
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple_rsi = self.tp_multiple_rsi 
        tp_price = planned_entry_price + per_share_risk * tp_multiple_rsi
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        # print(candle_date)
        # print(f"long per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go long limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.buy(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
            self.current_day_order_placed = True
        
    def place_short_order(self, candle_date):
        planned_entry_price = self.data.Low[-1]
        sl_price = self.last_high
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple_rsi = self.tp_multiple_rsi 
        tp_price = planned_entry_price - per_share_risk * tp_multiple_rsi
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        # print(candle_date)
        # print(f"short per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go short limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.sell(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
            self.current_day_order_placed = True
            
    def use_rsi_entry_strategy_market_ranging(self, candle_datetime):
        if self.last_low is None:
            self.last_low = self.data.Low[-1]
        else:
            if self.last_low > self.data.Low[-1]:
                self.last_low = self.data.Low[-1]

        if self.last_high is None:
            self.last_high = self.data.High[-1]
        else:
            if self.last_high < self.data.High[-1]:
                self.last_high = self.data.High[-1]

        c_long = self.data['RSI14'][-1] < 35
        c_short = self.data['RSI14'][-1] > 65
        c_time_limit = candle_datetime.time().hour < self.stop_trade_hour and candle_datetime.time().hour > self.start_trade_hour

        if not self.current_day_order_placed and c_time_limit:
            # print(f"not placed RSI14 = {self.data['RSI14'][-1]}")
            if c_long:
                self.place_long_order(candle_datetime)

            if c_short:
                self.place_short_order(candle_datetime)
        else:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            # print(f"placed RSI14 = {self.data['RSI14'][-1]}")
            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()
                        self.current_day_order_placed = False
                        if order.is_long:
                            self.place_long_order(candle_datetime)
                        else:
                            self.place_short_order(candle_datetime)

    def use_orb_strategy_market_trending(self, candle_datetime):
        if candle_datetime.time().hour < self.close_range_hour:
            return
        
        if candle_datetime.time().hour > self.close_range_hour and self.opening_range_high is not None and self.opening_range_low is not None:
            c_long = self.data.Open[-1] < self.opening_range_high and self.data.Close[-1] > self.opening_range_high and self.data['market_state'][-1] == 'BULLISH'
            c_short = self.data.Open[-1] > self.opening_range_low and self.data.Close[-1] < self.opening_range_low and self.data['market_state'][-1] == 'BEARISH'
            
            c2 = (self.opening_range_high - self.opening_range_low) < self.data['ATR'][-1] * 2.5
            c3 = (self.opening_range_high - self.opening_range_low) > self.data['ATR'][-1] * 1.0
            c4 = candle_datetime.time().hour < self.stop_trade_hour
            c5 = self.data['ER'][-1] > self.er15_threshold

            if not self.current_day_order_placed and c2 and c3 and c4 and c5:
                if c_long :
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_low
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_multiple = self.tp_multiple_orb 

                    tp_price = planned_entry_price + per_share_risk * tp_multiple
                    position_size = int((self.risk_percent * self.equity) / per_share_risk)

                    # print(f'Go long tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.buy(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_order_placed = True
                if c_short:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_high
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_multiple = self.tp_multiple_orb
                        
                    tp_price = planned_entry_price - per_share_risk * tp_multiple
                    position_size = int((self.risk_percent * self.equity) / per_share_risk)

                    # print(f'Go short tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.sell(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_order_placed = True

    def next(self):
        t = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()
        if (self.current_day != current_bar_date):
            self.reset_range(current_bar_date)

        if self.open_range_hour < t.time().hour and t.time().hour <= self.close_range_hour:
            if self.opening_range_high is not None:
                self.opening_range_high = max(self.opening_range_high, self.data.High[-1])
            else:
                self.opening_range_high = self.data.High[-1]

            if self.opening_range_low is not None:
                    self.opening_range_low = min(self.opening_range_low, self.data.Low[-1])
            else:
                self.opening_range_low = self.data.Low[-1]

        if self.data['is_range_box'][-1] == True:
            self.use_rsi_entry_strategy_market_ranging(t)
        else :
            self.use_orb_strategy_market_trending(t)

        # if self.data['ER'][-1] > self.er15_threshold:
        #     self.use_orb_strategy(t)
                            
        # print("--------------------------------")
            



In [7]:
from matplotlib import pyplot as plt
import pandas as pd
from backtesting import Backtest
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator

# all strategy
window = 2
global_er15_threshold = 0.3

# rsi
global_start_trade_hour = 5
global_tp_multiple_rsi = 5
global_stop_trade_rsi = 16

# orb
global_range_to_consider = 3
global_close_range_hour = 7
global_tp_multiple_orb = 5
global_stop_trade_orb = 18

# # start_date = '2020-11-1 00:00'
# end_date = '2023-2-28 00:00'
start_date = '2020-01-1 00:00'
end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"

# range box
window_rangebox = 15
touch_tol_pct = 0.15
min_touches = 3
overlap_threshold = 0.5
position_threshold = 0.2

df_d1 = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

df_range_box_list = [False for i in range(len(df_d1))]

for index in range(window_rangebox, len(df_d1)):
    start_index = index - window_rangebox
    range_ok, box_info = is_range_box(df_d1['High'][start_index:index], df_d1['Low'][start_index:index], df_d1['Close'][start_index:index], 
                                    N=window_rangebox, touch_tol_pct=touch_tol_pct, min_touches=min_touches, overlap_threshold=overlap_threshold, position_threshold=position_threshold)
                                
    df_range_box_list[index] = range_ok

df_d1['is_range_box'] = df_range_box_list

indicator_er = kaufman_efficiency_ratio(df_d1['Close'], 20)
df_result = implement_market_structure_states_strict(df_d1, n=window)
df_result['ER'] = indicator_er.values

df_h1 = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

h1_tf_direction_list = []
h1_tf_is_rangebox_list = []
h1_tf_er15_list = []

for i in range(len(df_result)):
    row = df_result.iloc[i]
    
    df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
    for index2, row2 in df_by_day.iterrows():
        h1_tf_direction_list.append(df_result.iloc[i-1]['market_state'])
        h1_tf_is_rangebox_list.append(df_result.iloc[i-1]['is_range_box'])
        h1_tf_er15_list.append(df_result.iloc[i-1]['ER'])

df_h1['market_state'] = h1_tf_direction_list
df_h1['is_range_box'] = h1_tf_is_rangebox_list
df_h1['ER'] = h1_tf_er15_list

indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr

indicator_rsi_14 = RSIIndicator(df_h1['Close'], window=14)
df_h1['RSI14'] = indicator_rsi_14._rsi
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-07-29 14:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2020-11-10 07:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

Start                     2020-01-01 22:00...
End                       2026-09-16 01:00...
Duration                   2449 days 02:00:00
Exposure Time [%]                    18.30528
Equity Final [$]                    25355.968
Equity Peak [$]                     30111.967
Return [%]                          153.55968
Buy & Hold Return [%]                45.36014
Return (Ann.) [%]                    11.83017
Volatility (Ann.) [%]                17.22313
CAGR [%]                             14.88504
Sharpe Ratio                          0.68688
Sortino Ratio                          1.4518
Calmar Ratio                          0.28484
Alpha [%]                           144.43763
Beta                                   0.2011
Max. Drawdown [%]                   -41.53203
Avg. Drawdown [%]                    -2.20028
Max. Drawdown Duration     1511 days 07:00:00
Avg. Drawdown Duration       23 days 20:00:00
# Trades                                  265
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p1261', ...)

In [ ]:
# test variables
import time

from matplotlib import pyplot as plt
import pandas as pd
from backtesting import Backtest
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator

# all strategy
window = 2
global_stop_trade_hour = 16
global_er15_threshold = 0.3

# rsi
global_start_trade_hour = 5
global_tp_multiple_rsi = 4

# orb
global_range_to_consider = 3
global_close_range_hour = 7
global_tp_multiple_orb = 5

start_date = '2020-11-1 00:00'
end_date = '2023-2-28 00:00'
# start_date = '2020-11-1 00:00'
# end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"

df_d1 = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]



indicator_er = kaufman_efficiency_ratio(df_d1['Close'], 20)
df_result = implement_market_structure_states_strict(df_d1, n=window)
df_result['ER'] = indicator_er.values

df_h1 = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

h1_tf_direction_list = []
h1_tf_er15_list = []

for i in range(len(df_result)):
    row = df_result.iloc[i]
    
    df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
    for index2, row2 in df_by_day.iterrows():
        h1_tf_direction_list.append(df_result.iloc[i-1]['market_state'])
        h1_tf_er15_list.append(df_result.iloc[i-1]['ER'])

df_h1['market_state'] = h1_tf_direction_list
df_h1['ER'] = h1_tf_er15_list

indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr

indicator_rsi_14 = RSIIndicator(df_h1['Close'], window=14)
df_h1['RSI14'] = indicator_rsi_14._rsi
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

window_rangebox_list = [15,20,25]
touch_tol_pct_list = [0.05,0.1,0.15]
min_touches_list = [2,3,4]
overlap_threshold_list = [0.5,0.6,0.7]
position_threshold_list = [0.2,0.25,0.35]
tp_multiple_list = [1,2,3,4,5,6,7,8,9,10]

with open(f"./results/result_{pair}_{time.time()}.csv", "a") as f:
    f.write(f"window_rangebox,touch_tol_pct,min_touches,overlap_threshold,position_threshold,tp_multiple,Return,ReturnYearly,MDrawdown,WinRate,NumTrades\n")

    for window_rangebox in window_rangebox_list:
        for touch_tol_pct in touch_tol_pct_list:
            for min_touches in min_touches_list:
                for overlap_threshold in overlap_threshold_list:
                    for position_threshold in position_threshold_list:
                        for tp_multiple in tp_multiple_list:
                            global_tp_multiple_rsi = tp_multiple
                            df_range_box_list = [False for i in range(len(df_d1))]

                            for index in range(window_rangebox, len(df_d1)):
                                start_index = index-window_rangebox
                                range_ok, box_info = is_range_box(df_d1['High'][start_index:index], df_d1['Low'][start_index:index], df_d1['Close'][start_index:index], 
                                                                N=window_rangebox, touch_tol_pct=touch_tol_pct, min_touches=min_touches, overlap_threshold=overlap_threshold, position_threshold=position_threshold)
                                df_range_box_list[index] = range_ok
                            df_d1['is_range_box'] = df_range_box_list

                            h1_tf_is_rangebox_list = []
                            for i in range(len(df_d1)):
                                row = df_d1.iloc[i]
                                
                                df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
                                for index2, row2 in df_by_day.iterrows():
                                    h1_tf_is_rangebox_list.append(df_d1.iloc[i-1]['is_range_box'])
                            df_h1['is_range_box'] = h1_tf_is_rangebox_list

                            bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)
                            stats = bt.run()
                    
                            f.write(f"{window_rangebox},{touch_tol_pct},{min_touches},{overlap_threshold},{position_threshold},{tp_multiple},{stats['Return [%]']},{stats['Return (Ann.) [%]']},{stats['Max. Drawdown [%]']},{stats['Win Rate [%]']}, {stats['# Trades']}\n")
                    
                            print(f"window_rangebox = {window_rangebox} & touch_tol_pct = {touch_tol_pct} & min_touches = {min_touches} & overlap_threshold = {overlap_threshold} & position_threshold = {position_threshold} & tp_multiple = {tp_multiple} DONE")


window_rangebox = 15 & touch_tol_pct = 0.05 & min_touches = 2 & overlap_threshold = 0.5 & position_threshold = 0.2 & tp_multiple = 1 DONE
window_rangebox = 15 & touch_tol_pct = 0.05 & min_touches = 2 & overlap_threshold = 0.5 & position_threshold = 0.2 & tp_multiple = 2 DONE
window_rangebox = 15 & touch_tol_pct = 0.05 & min_touches = 2 & overlap_threshold = 0.5 & position_threshold = 0.2 & tp_multiple = 3 DONE
window_rangebox = 15 & touch_tol_pct = 0.05 & min_touches = 2 & overlap_threshold = 0.5 & position_threshold = 0.2 & tp_multiple = 4 DONE
window_rangebox = 15 & touch_tol_pct = 0.05 & min_touches = 2 & overlap_threshold = 0.5 & position_threshold = 0.2 & tp_multiple = 5 DONE
window_rangebox = 15 & touch_tol_pct = 0.05 & min_touches = 2 & overlap_threshold = 0.5 & position_threshold = 0.2 & tp_multiple = 6 DONE
window_rangebox = 15 & touch_tol_pct = 0.05 & min_touches = 2 & overlap_threshold = 0.5 & position_threshold = 0.2 & tp_multiple = 7 DONE
window_rangebox = 15 & touch_tol_p

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2022-12-29 15:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(


window_rangebox = 15 & touch_tol_pct = 0.05 & min_touches = 2 & overlap_threshold = 0.5 & position_threshold = 0.35 & tp_multiple = 1 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2022-12-29 15:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(


window_rangebox = 15 & touch_tol_pct = 0.05 & min_touches = 2 & overlap_threshold = 0.5 & position_threshold = 0.35 & tp_multiple = 2 DONE


...

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2021-03-24 07:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2021-10-01 08:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

window_rangebox = 25 & touch_tol_pct = 0.1 & min_touches = 3 & overlap_threshold = 0.5 & position_threshold = 0.25 & tp_multiple = 8 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2021-03-24 07:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2021-10-01 08:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

window_rangebox = 25 & touch_tol_pct = 0.1 & min_touches = 3 & overlap_threshold = 0.5 & position_threshold = 0.25 & tp_multiple = 9 DONE


KeyboardInterrupt: 